# Chart Esai — ForestWatch Papua (skema 7 kelas)

Baca langsung output training 7-kelas dari Google Drive (`metrics_finetune.json`,
`train_distribution.json`, `summary_finetune.json`, `best_model_finetune_resume_v2.pt`),
hasilkan semua chart untuk esai. Jalankan di Colab dari atas ke bawah. Chart tersimpan PNG di
Drive, folder yang sama dgn output training (`.../Bahan_Training_Fix_Combined_v4/output/chart_esai/`).

Chart yang dihasilkan:
1. Distribusi kelas (ketidakseimbangan data, 7 kelas)
2. IoU per kelas (7 kelas)
3. Confusion matrix 7x7 (ternormalisasi)
4. Panel metrik utama — **2 gambar**: OA+Kappa+FWIoU (gabung), mIoU (sendiri)
5. (opsional) IoU 7-kelas: sebelum vs sesudah fine-tune
6. (opsional) Kurva training per-epoch (loss + val mIoU) — **bukan FWIoU**: FWIoU butuh
   confusion matrix lengkap dan hanya dihitung SEKALI di akhir (TEST), tak pernah dihitung
   per-epoch saat training (lihat catatan di cell-nya).


In [ ]:
# === Bagian 0 -- Mount Drive + lokasi file output training 7-kelas ===
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

# SESUAIKAN kalau folder output-mu beda lokasi.
OUTPUT_DIR = Path("/content/drive/MyDrive/Satria Data 3.0/Bahan_Training_Fix_Combined_v4/output")
METRICS_JSON = OUTPUT_DIR / "metrics_finetune.json"
DIST_JSON    = OUTPUT_DIR / "train_distribution.json"

CHART_DIR = OUTPUT_DIR / "chart_esai"
CHART_DIR.mkdir(parents=True, exist_ok=True)

assert METRICS_JSON.exists(), f"Tak ditemukan: {METRICS_JSON}"
assert DIST_JSON.exists(), f"Tak ditemukan: {DIST_JSON}"
print("METRICS_JSON :", METRICS_JSON)
print("DIST_JSON    :", DIST_JSON)
print("Chart disimpan ke:", CHART_DIR)


In [ ]:
# === Bagian 1 -- Load data asli (7-kelas) + setup ===
import json
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.axisbelow": True, "figure.dpi": 120,
                     "savefig.dpi": 150, "savefig.bbox": "tight"})

metrics7 = json.load(open(METRICS_JSON))
dist7_raw = json.load(open(DIST_JSON))
dist7 = {int(k): int(v) for k, v in dist7_raw.items()}

NAMES7 = ["Perairan", "Hutan", "Lahan Terbuka", "Sawit", "Pertanian Lain", "Tambang", "Permukiman"]
COLORS7 = ["#2A6FDB", "#0B3D0B", "#E03B24", "#F97316", "#E9C46A", "#8E24AA", "#757575"]

cm7 = np.array(metrics7["confusion_matrix"], dtype=np.int64)
iou7 = [r["iou"] for r in metrics7["per_class"]]
oa7, kappa7, fwiou7, miou7 = (metrics7["overall_accuracy"], metrics7["kappa"],
                              metrics7["fwiou"], metrics7["mean_iou"])

print(f"OA={oa7:.4f} | Kappa={kappa7:.4f} | mIoU={miou7:.4f} | FWIoU={fwiou7:.4f}")
print("Distribusi train (7-kelas mentah):", dist7)


In [ ]:
# === Bagian 2 -- (opsional) summary_finetune.json utk IoU SEBELUM fine-tune (chart 5) ===
SUMMARY_JSON = OUTPUT_DIR / "summary_finetune.json"
iou7_before = None
if SUMMARY_JSON.exists():
    summary7 = json.load(open(SUMMARY_JSON))
    iou7_before = summary7.get("baseline_test_per_class_iou")
    print("baseline_test_per_class_iou (sebelum fine-tune):", iou7_before)
else:
    print("summary_finetune.json tak ditemukan -- Chart 5 (before/after) akan dilewati.")


## Chart 1 — Distribusi kelas (ketidakseimbangan data, 7 kelas)
Dipakai di **Pendahuluan/Metodologi** untuk menjelaskan tantangan kelas minoritas.

In [ ]:
# === Chart 1: Distribusi piksel kelas (train, 7 kelas) ===
tot = sum(dist7.values())
pct = [100 * dist7.get(c, 0) / tot for c in range(7)]
order = np.argsort(pct)  # kecil -> besar
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh([NAMES7[i] for i in order], [pct[i] for i in order],
        color=[COLORS7[i] for i in order], edgecolor="black", linewidth=0.5)
for i, idx in enumerate(order):
    ax.text(pct[idx] + 0.4, i, f"{pct[idx]:.1f}%", va="center", fontsize=10)
ax.set_xlabel("Proporsi piksel di data latih (%)")
ax.set_title("Distribusi Kelas Tutupan Lahan (7 kelas) - ketidakseimbangan kuat")
ax.set_xlim(0, max(pct) * 1.15); ax.grid(axis="y", visible=False)
fig.savefig(CHART_DIR / "01_distribusi_kelas.png"); plt.show()
print("Tersimpan:", CHART_DIR / "01_distribusi_kelas.png")


## Chart 2 — IoU per kelas (7 kelas)
Chart performa utama. Dipakai di **Pembahasan** (hasil per kelas).

In [ ]:
# === Chart 2: IoU per-kelas (7 kelas, model final) ===
fig, ax = plt.subplots(figsize=(9, 4.8))
bars = ax.bar(NAMES7, iou7, color=COLORS7, edgecolor="black", linewidth=0.5)
for b, v in zip(bars, iou7):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.3f}", ha="center", fontsize=10)
ax.axhline(miou7, color="black", ls="--", lw=1.3, label=f"mIoU = {miou7:.3f}")
ax.set_ylabel("IoU"); ax.set_ylim(0, 1.05); ax.grid(axis="x", visible=False)
ax.set_title("IoU per Kelas - Model Final (7 kelas, TEST set)")
ax.legend(); plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
fig.savefig(CHART_DIR / "02_iou_per_kelas.png"); plt.show()
print("Tersimpan:", CHART_DIR / "02_iou_per_kelas.png")


## Chart 3 — Confusion matrix 7 kelas (ternormalisasi)
Untuk analisis kesalahan di **Pembahasan**. Diagonal = benar; sel lain = tertukar.

In [ ]:
# === Chart 3: Confusion matrix 7x7 (ternormalisasi per baris) ===
cmn = cm7 / cm7.sum(1, keepdims=True).clip(1)
fig, ax = plt.subplots(figsize=(7.8, 6.6))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(7):
    for j in range(7):
        ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=9,
                color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xticks(range(7)); ax.set_xticklabels(NAMES7, rotation=40, ha="right")
ax.set_yticks(range(7)); ax.set_yticklabels(NAMES7)
ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual (ground truth)")
ax.set_title("Confusion Matrix - 7 kelas (ternormalisasi per baris)")
ax.grid(False); fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.savefig(CHART_DIR / "03_confusion_matrix.png"); plt.show()
print("Tersimpan:", CHART_DIR / "03_confusion_matrix.png")


## Chart 4 — Panel metrik utama (2 GAMBAR)
OA + Kappa + FWIoU digabung satu chart ("metrik agregat", semuanya dipengaruhi kelas
mayoritas); mIoU dipisah sendiri (metrik per-kelas tak berbobot, paling jujur soal kelas
minoritas) — lihat penjelasan bedanya di teks setelah cell ini.

In [ ]:
# === Chart 4a: Overall Accuracy + Kappa + FWIoU (digabung 1 gambar) ===
fig, ax = plt.subplots(figsize=(6, 4.6))
labels_a = ["Overall\nAccuracy", "Cohen's\nKappa", "FWIoU"]
vals_a = [oa7, kappa7, fwiou7]
cols_a = ["#2E7D32", "#1565C0", "#6A1B9A"]
bars = ax.bar(labels_a, vals_a, color=cols_a, edgecolor="black", linewidth=0.7, width=0.55)
for b, v in zip(bars, vals_a):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.3f}",
            ha="center", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.08); ax.set_ylabel("Nilai"); ax.grid(axis="x", visible=False)
ax.set_title("Metrik Evaluasi Agregat (7 kelas, TEST set)")
fig.savefig(CHART_DIR / "04a_oa_kappa_fwiou.png"); plt.show()
print("Tersimpan:", CHART_DIR / "04a_oa_kappa_fwiou.png")

# === Chart 4b: mIoU (terpisah sendiri) ===
fig, ax = plt.subplots(figsize=(3.6, 4.4))
ax.bar(["mIoU"], [miou7], color="#F9A825", edgecolor="black", linewidth=0.7, width=0.55)
ax.text(0, miou7 + 0.025, f"{miou7:.3f}", ha="center", fontsize=15, fontweight="bold")
ax.set_ylim(0, 1.08); ax.set_ylabel("Nilai")
ax.set_title("mIoU", fontsize=13)
ax.grid(axis="x", visible=False)
fig.savefig(CHART_DIR / "04b_miou.png"); plt.show()
print("Tersimpan:", CHART_DIR / "04b_miou.png")


## Chart 5 (opsional) — IoU 7 kelas: sebelum vs sesudah fine-tune
Cerita **pengembangan model** (fine-tune menaikkan tiap kelas, terutama kelas minoritas). Butuh `summary_finetune.json` (Bagian 2) -- dilewati otomatis kalau tak ada.

In [ ]:
# === Chart 5: IoU per-kelas 7-kelas, sebelum vs sesudah fine-tune ===
if iou7_before is None:
    print("Dilewati -- summary_finetune.json tak ditemukan (lihat Bagian 2).")
else:
    x = np.arange(7); w = 0.38
    fig, ax = plt.subplots(figsize=(10, 4.8))
    b1 = ax.bar(x - w / 2, iou7_before, w, label="Sebelum fine-tune",
                color="#B0BEC5", edgecolor="black", linewidth=0.4)
    b2 = ax.bar(x + w / 2, iou7, w, label="Sesudah fine-tune",
                color="#1565C0", edgecolor="black", linewidth=0.4)
    for bars in (b1, b2):
        for b in bars:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.012,
                    f"{b.get_height():.2f}", ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(NAMES7, rotation=20, ha="right")
    ax.set_ylabel("IoU"); ax.set_ylim(0, 1.08); ax.grid(axis="x", visible=False)
    ax.set_title("Pengembangan Model 7 Kelas: IoU Sebelum vs Sesudah Fine-tune (TEST)")
    ax.legend()
    fig.savefig(CHART_DIR / "05_sebelum_sesudah.png"); plt.show()
    print("Tersimpan:", CHART_DIR / "05_sebelum_sesudah.png")


## Chart 6 (opsional) — Kurva training per-epoch (loss + val mIoU)

**Catatan jujur**: ini BUKAN kurva FWIoU. Selama training, tiap epoch cuma `val mIoU`
(`MulticlassJaccardIndex`, IoU rata-rata tak berbobot) yang dihitung dan disimpan ke
`history` -- lihat `model/src/forestwatch/training/trainer.py`. **FWIoU butuh confusion
matrix lengkap** (dibobot frekuensi piksel tiap kelas) dan di notebook training itu **hanya
dihitung SEKALI**, di akhir, pas evaluasi TEST final -- itulah satu-satunya angka FWIoU yang
ada (sudah di `metrics_finetune.json`, dipakai Chart 4a). Checkpoint per-epoch juga tak
disimpan satu-satu (cuma checkpoint TERBAIK + checkpoint TERAKHIR/resume yang ter-overwrite
tiap epoch), jadi FWIoU per-epoch **tak bisa direkonstruksi** dari data yang ada.

Yang BISA ditampilkan jujur: kurva `train_loss`/`val_loss` dan `val_miou` per-epoch, dari
`history` yang tersimpan di dalam `best_model_finetune_resume_v2.pt`.

In [ ]:
# === Chart 6: Kurva training per-epoch (loss + val mIoU) -- dari history resume checkpoint ===
import torch  # Colab sudah preinstall torch -- tak perlu paket forestwatch

RESUME_PT = OUTPUT_DIR / "best_model_finetune_resume_v2.pt"
if not RESUME_PT.exists():
    print(f"Dilewati -- {RESUME_PT} tak ditemukan.")
else:
    state = torch.load(RESUME_PT, map_location="cpu")
    history = state.get("history")
    if not history:
        print("Dilewati -- key 'history' kosong/tak ada di checkpoint ini.")
    else:
        eps = [h["epoch"] for h in history]
        tr_loss = [h["train_loss"] for h in history]
        vl_loss = [h["val_loss"] for h in history]
        vl_miou = [h["val_miou"] for h in history]

        fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
        ax[0].plot(eps, tr_loss, label="train_loss", lw=2)
        ax[0].plot(eps, vl_loss, label="val_loss", lw=2)
        ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss")
        ax[0].set_title("Loss per Epoch"); ax[0].legend()

        ax[1].plot(eps, vl_miou, color="#F9A825", lw=2.2, label="val mIoU")
        ax[1].axhline(miou7, color="black", ls="--", lw=1.2, label=f"mIoU TEST final = {miou7:.3f}")
        ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("mIoU"); ax[1].set_ylim(0, 1)
        ax[1].set_title("val mIoU per Epoch (BUKAN FWIoU)"); ax[1].legend()
        fig.suptitle(f"Kurva Training -- {len(history)} epoch tercatat")
        fig.tight_layout()
        fig.savefig(CHART_DIR / "06_kurva_training_per_epoch.png"); plt.show()
        print("Tersimpan:", CHART_DIR / "06_kurva_training_per_epoch.png")
        print(f"\nEpoch terbaik (val mIoU tertinggi): ep{eps[int(np.argmax(vl_miou))]:02d} "
              f"(val mIoU={max(vl_miou):.4f})")
